# Employee Salary Prediction — Linear Regression, Decision Tree & Random Forest

This notebook uses the SalaryPredictions dataset and applies **only**:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor

Target: `salary`

`jobId` is removed because it is an identifier. Categorical predictors are one-hot encoded. The three models are evaluated on the same held-out test set using **MAE, RMSE, and R²**.

> For faster execution on a normal laptop, `NROWS = 100_000` by default. Set `NROWS = None` to use the full 1,000,000-row training dataset.

### Repository note
The raw dataset is intentionally not committed to this repository. Download `SalaryPredictions.zip`
from the dataset source listed in the README and place it inside the `data/` folder before running
the notebook. The notebook uses 100,000 rows by default for practical runtime; set `NROWS = None`
to use the complete training data.


In [ ]:
from pathlib import Path
import zipfile
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
NROWS = 100_000


## 1. Load the dataset

In [ ]:
# Works with either extracted CSV files or SalaryPredictions.zip.

possible_feature_files = [
    Path("data/train_features.csv"),
    Path("../data/train_features.csv"),
    Path("train_features.csv"),
    Path("/mnt/data/salary_prediction_1m_data/data/train_features.csv"),
]

possible_salary_files = [
    Path("data/train_salaries.csv"),
    Path("../data/train_salaries.csv"),
    Path("train_salaries.csv"),
    Path("/mnt/data/salary_prediction_1m_data/data/train_salaries.csv"),
]

features_path = next((p for p in possible_feature_files if p.exists()), None)
salary_path = next((p for p in possible_salary_files if p.exists()), None)

if features_path is None or salary_path is None:
    zip_candidates = [
        Path("data/SalaryPredictions.zip"),
        Path("SalaryPredictions.zip"),
        Path("../SalaryPredictions.zip"),
        Path("/mnt/data/SalaryPredictions.zip"),
    ]
    zip_path = next((p for p in zip_candidates if p.exists()), None)

    if zip_path is None:
        raise FileNotFoundError(
            "Place SalaryPredictions.zip beside this notebook, "
            "or provide train_features.csv and train_salaries.csv."
        )

    extract_dir = Path("salary_prediction_data")
    extract_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    features_path = extract_dir / "data" / "train_features.csv"
    salary_path = extract_dir / "data" / "train_salaries.csv"

features = pd.read_csv(features_path, nrows=NROWS)
salaries = pd.read_csv(salary_path, nrows=NROWS)

data = features.merge(salaries, on="jobId", how="inner")

print("Dataset shape:", data.shape)
print("Missing values:", int(data.isna().sum().sum()))
print("Zero salary rows:", int((data["salary"] == 0).sum()))
display(data.head())


## 2. Data cleaning

In [ ]:
# Remove anomalous zero-salary rows.
before = len(data)
data = data[data["salary"] > 0].copy()

print("Removed rows:", before - len(data))
print("Rows used:", len(data))


## 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data["salary"], bins=40)
plt.xlabel("Salary")
plt.ylabel("Frequency")
plt.title("Salary Distribution")
plt.tight_layout()
plt.show()


In [ ]:
job_salary = (
    data.groupby("jobType", observed=False)["salary"]
        .mean()
        .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
plt.bar(job_salary.index, job_salary.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Average Salary")
plt.title("Average Salary by Job Type")
plt.tight_layout()
plt.show()

display(job_salary.to_frame("mean_salary"))


In [ ]:
sample = data.sample(min(10_000, len(data)), random_state=RANDOM_STATE)

plt.figure(figsize=(8, 5))
plt.scatter(sample["yearsExperience"], sample["salary"], alpha=0.25)
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Salary vs Years of Experience")
plt.tight_layout()
plt.show()


## 4. Features and train/test split

In [ ]:
target = "salary"

# Remove jobId because it is only a unique identifier.
X = data.drop(columns=[target, "jobId"])
y = data[target]

categorical_features = [
    "companyId",
    "jobType",
    "degree",
    "major",
    "industry",
]

numeric_features = [
    "yearsExperience",
    "milesFromMetropolis",
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


## 5. Preprocessing

In [ ]:
preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features,
    ),
    (
        "numeric",
        "passthrough",
        numeric_features,
    ),
])

print("Categorical:", categorical_features)
print("Numerical:", numeric_features)


## 6. Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=80,
        max_depth=16,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

models


## 7. Train and evaluate all three models

In [ ]:
results = []
fitted_models = {}
predictions_by_model = {}

for name, estimator in models.items():
    print(f"Training {name}...")
    start = time.time()

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Training_seconds": time.time() - start,
    })

    fitted_models[name] = pipeline
    predictions_by_model[name] = pred

results_df = (
    pd.DataFrame(results)
      .sort_values("R2", ascending=False)
      .reset_index(drop=True)
)

display(results_df)


## 8. Compare R²

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["R2"])
plt.ylabel("Test R²")
plt.title("Model Comparison")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_predictions = predictions_by_model[best_model_name]

print("Best model:", best_model_name)
print(results_df.iloc[0])


## 9. Actual vs predicted

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.25)

low = min(float(y_test.min()), float(best_predictions.min()))
high = max(float(y_test.max()), float(best_predictions.max()))

plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Salary")
plt.ylabel("Predicted Salary")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()


## 10. Residual plot

In [ ]:
residuals = y_test.to_numpy() - best_predictions

plt.figure(figsize=(7, 5))
plt.scatter(best_predictions, residuals, alpha=0.25)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Salary")
plt.ylabel("Residual")
plt.title(f"Residual Plot — {best_model_name}")
plt.tight_layout()
plt.show()


## 11. Conclusion

The three requested models are compared on exactly the same train/test split.

- **Linear Regression** provides a simple and interpretable baseline.
- **Decision Tree** captures nonlinear effects but is controlled using depth and minimum-leaf restrictions.
- **Random Forest** combines many decision trees to reduce variance and improve generalization.

The final table should be used to report the actual MAE, RMSE, and R² obtained after running the notebook.
